In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("Indian_Kids_Screen_Time.csv")
df.head()

,Age,Gender,Avg_Daily_Screen_Time_hr,Primary_Device,Exceeded_Recommended_Limit,Educational_to_Recreational_Ratio,Health_Impacts,Urban_or_Rural
0,14,Male,3.99,Smartphone,True,0.42,"Poor Sleep, Eye Strain",Urban
1,11,Female,4.61,Laptop,True,0.30,Poor Sleep,Urban
2,18,Female,3.73,TV,True,0.32,Poor Sleep,Urban
3,15,Female,1.21,Laptop,False,0.39,NaN,Urban
4,12,Female,5.89,Smartphone,True,0.49,"Poor Sleep, Anxiety",Urban


In [3]:
x=df.drop(['Avg_Daily_Screen_Time_hr'],axis=1)
y=df['Avg_Daily_Screen_Time_hr']

In [4]:
x.isnull().sum()

Age                                     0
Gender                                  0
Primary_Device                          0
Exceeded_Recommended_Limit              0
Educational_to_Recreational_Ratio       0
Health_Impacts                       3218
Urban_or_Rural                          0
dtype: int64

In [5]:
x['Health_Impacts']=x['Health_Impacts'].fillna(x['Health_Impacts'].mode()[0])

In [6]:
x.isnull().sum()

Age                                  0
Gender                               0
Primary_Device                       0
Exceeded_Recommended_Limit           0
Educational_to_Recreational_Ratio    0
Health_Impacts                       0
Urban_or_Rural                       0
dtype: int64

In [7]:
num_cols = x.select_dtypes(include=['int64', 'float64']).columns

def cap_outliers(x, cols):
    x = x.copy()
    
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        x[col] = x[col].clip(lower, upper)
    
    return x
x = cap_outliers(x, num_cols)

In [8]:
cat_cols= x.select_dtypes(include=['object']).columns

In [9]:
#apply columntransformer 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,StandardScaler
ct=ColumnTransformer(transformers=[("oe",OrdinalEncoder(),cat_cols),
                                   ("sc",StandardScaler(),num_cols)])

In [10]:
x_transformed=ct.fit_transform(x)

In [11]:
x_transformed=pd.DataFrame(x_transformed)
x_transformed

,0,1,2,3,4,5
0,1.0,1.0,10.0,1.0,0.322805,-0.098694
1,0.0,0.0,7.0,1.0,-0.625879,-1.737647
2,0.0,2.0,7.0,1.0,1.587718,-1.464488
3,0.0,0.0,7.0,1.0,0.639033,-0.508432
4,0.0,1.0,8.0,1.0,-0.309651,0.857362
...,...,...,...,...,...,...
9707,1.0,1.0,7.0,1.0,1.271490,0.174465
9708,0.0,1.0,7.0,0.0,1.271490,-0.371853
9709,1.0,1.0,11.0,0.0,0.955262,-0.508432
9710,1.0,2.0,7.0,1.0,1.271490,0.037886


In [12]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x_transformed,y,test_size=0.2)

In [13]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
dt.fit(x_train,y_train)

DecisionTreeRegressor(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                      random_state=42)

In [14]:
params = {
    "max_depth": [3, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 6],
    "criterion": ["squared_error", "friedman_mse"],
    "splitter": ["best", "random"]
}

In [15]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(
    estimator=dt,
    param_grid=params,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)


In [16]:
grid.fit(x_train, y_train)

Fitting 5 folds for each of 320 candidates, totalling 1600 fits


GridSearchCV(cv=5,
             estimator=DecisionTreeRegressor(max_depth=10, min_samples_leaf=2,
                                             min_samples_split=5,
                                             random_state=42),
             n_jobs=-1,
             param_grid={'criterion': ['squared_error', 'friedman_mse'],
                         'max_depth': [3, 5, 10, 15, 20],
                         'min_samples_leaf': [1, 2, 4, 6],
                         'min_samples_split': [2, 5, 10, 20],
                         'splitter': ['best', 'random']},
             scoring='r2', verbose=2)

In [17]:
print("Best Parameters:")
print(grid.best_params_)

print("\nBest R2 Score:")
print(grid.best_score_)

Best Parameters:
{'criterion': 'squared_error', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2, 'splitter': 'best'}

Best R2 Score:
0.14447487290064986


In [18]:
best_dt = grid.best_estimator_

In [19]:
y_pred = best_dt.predict(x_test)


In [20]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("\n===== FINAL MODEL PERFORMANCE =====")

print("R2 Score :", r2)
print("MAE      :", mae)
print("MSE      :", mse)
print("RMSE     :", rmse)




===== FINAL MODEL PERFORMANCE =====
R2 Score : 0.09334001178196283
MAE      : 1.2060503440744181
MSE      : 2.5567101208769674
RMSE     : 1.5989715822606
